In [ ]:
import csv
import urllib.request
import re
from datetime import datetime

# 1. Definición de la función de lectura
def carregar_dados(caminho_arquivo):
    """
    Abre qualquer arquivo CSV local e retorna seus dados como lista de dicionários.
    """
    with open(caminho_arquivo, mode="r", encoding="utf-8") as f:
        leitor = csv.DictReader(f)
        return list(leitor)

        # URLs dos datasets
url_produtos = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/refs/heads/main/olist_products_dataset.csv"
url_pedidos = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/refs/heads/main/olist_orders_dataset.csv"

# 2. Descarga de los archivos en Colab
urllib.request.urlretrieve(url_produtos, "productos.csv")
urllib.request.urlretrieve(url_pedidos, "pedidos.csv")

# 3. Lectura modular usando la función
productos = carregar_dados("productos.csv")
pedidos = carregar_dados("pedidos.csv")

# 4. Comprobación
print("--- PRIMER PRODUCTO ---")
print(productos[0])

print("\n--- PRIMER PEDIDO ---")
print(pedidos[0])

1. Validação e Tratamento de Dados Ausentes

In [ ]:

for producto in productos:
    categoria = (producto['product_category_name'] or '').strip()
    producto['product_category_name'] = categoria if categoria else 'sem categoria'

    # Contamos cuántos productos quedaron etiquetados como 'sem categoria'
sin_categoria = 0

for producto in productos:
    if producto['product_category_name'] == 'sem categoria':
        sin_categoria += 1

print(f"Total de productos asignados a 'sem categoria': {sin_categoria}")

In [ ]:
def organizar_produtos(lista_produtos):
    """
    Trata categorias ausentes/formatadas e remove registros com dimensões nulas.
    """
    colunas_dimensoes = [
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]

    produtos_limpios = []
    total_sem_categoria = 0

    for producto in lista_produtos:
        # 1. Filtro: manter apenas se todas as dimensões forem válidas
        if all((producto[col] or '').strip() for col in colunas_dimensoes):

            # 2. Tratamento da categoria (limpeza e substituição de nulos)
            categoria = (producto.get('product_category_name') or '').strip()
            if not categoria:
                producto['product_category_name'] = 'sem categoria'
                total_sem_categoria += 1
            else:
                # Padronização em minúsculas e remoção de pontuações
                categoria_limpa = re.sub(r'[^\w\s]', '', categoria.lower())
                producto['product_category_name'] = categoria_limpa

            produtos_limpios.append(producto)

    # Estatísticas do processamento
    total_original = len(lista_produtos)
    total_descartados = total_original - len(produtos_limpios)

    metricas = {
        'total_original': total_original,
        'total_validos': len(produtos_limpios),
        'total_descartados': total_descartados,
        'total_sem_categoria': total_sem_categoria
    }

    return produtos_limpios, metricas

# Chamada da função com a lista carregada
productos_limpios, metricas_produtos = organizar_produtos(productos)

# Auditoria dos resultados
print(f"Total original de produtos:       {metricas_produtos['total_original']}")
print(f"Total após remoção de nulos:      {metricas_produtos['total_validos']}")
print(f"Registros descartados:            {metricas_produtos['total_descartados']}")
print(f"Produtos como 'sem categoria':    {metricas_produtos['total_sem_categoria']}")

2. Padronização de Strings e Regex

In [ ]:
for p in productos_limpios:
   p['product_category_name'] = re.sub(r'[^\w\s]', '', p['product_category_name'].lower().strip())

   # Verificação de qualidade: imprime as primeiras 5 categorias limpas
print("--- AMOSTRA DE CATEGORIAS PROCESSADAS ---")
for p in productos_limpios[:5]:
    print(f"ID: {p['product_id']} | Categoria: '{p['product_category_name']}'")


In [ ]:
def sanitizar_produtos(lista_produtos):
    """Aplica regras de limpeza e padronização na base de produtos."""
    produtos_processados = []
    colunas_dimensoes = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

    for p in lista_produtos:
        # Descarta registros com dimensões nulas
        if all((p[col] or '').strip() for col in colunas_dimensoes):
            cat = p.get('product_category_name', '')
            if not cat or not cat.strip():
                p['product_category_name'] = 'sem categoria'
            else:
                p['product_category_name'] = re.sub(r'[^\w\s]', '', cat.lower().strip())
            produtos_processados.append(p)

    return produtos_processados

3. Lógica de Regra de Negócio

In [ ]:
# 1. Inicialização dos contadores
total_cancelados = 0
total_outros_status = 0

# 2. Iteração sobre a lista de pedidos
for pedido in pedidos:
    # Filtro: verifica se a data de entrega está ausente
    if not pedido['order_delivered_customer_date'].strip():
        if pedido['order_status'] == 'canceled':
            total_cancelados += 1
        else:
            total_outros_status += 1

# 3. Exibição dos resultados da hipótese
print("--- RESULTADO DA HIPÓTESE DE NEGÓCIO ---")
print(f"Pedidos sem data com status 'canceled': {total_cancelados}")
print(f"Pedidos sem data com outros status:      {total_outros_status}")

4. Formatação Temporal (Datetime):

In [ ]:
# 1. Definição da função de formatação
def formatar_data(data_string):
    """
    Objetivo: Converter string de data para o formato 'DD/MM/YYYY'.
    """
    if not data_string or not data_string.strip():
        return ""

    objeto_data = datetime.strptime(data_string.strip(), "%Y-%m-%d %H:%M:%S")
    return objeto_data.strftime("%d/%m/%Y")

# 2. Aplicação da transformação em toda a lista de pedidos
for pedido in pedidos:
    pedido['order_approved_at'] = formatar_data(pedido['order_approved_at'])

# 3. Amostra de validação dos resultados
print("--- AMOSTRA DE DATAS FORMATADAS (DD/MM/YYYY) ---")
for pedido in pedidos[:5]:
    print(f"ID: {pedido['order_id']} | Data de aprovação: {pedido['order_approved_at']}")

5. Relatório de Status Manual

In [ ]:
# --- CÁLCULO DAS MÉTRICAS ---
total_linhas = len(pedidos)
total_cancelados = sum(1 for p in pedidos if p['order_status'] == 'canceled')

# --- RELATÓRIO DE STATUS MANUAL ---
print('=' * 40)
print('         RELATÓRIO DE STATUS')
print('=' * 40)
print(f'Total de linhas processadas:      {total_linhas}')
print(f'Total de registros sem aprovação: {total_nulos_aprovacao}')
print(f'Total de pedidos cancelados:      {total_cancelados}')
print('=' * 40)
print('Status da base: SANITIZADA COM SUCESSO')